In [24]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import joblib 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [25]:
dataset = pd.read_csv("../../data/processed/Iris.csv")

In [26]:
target = "Species"
x = dataset.drop(columns=target)
y = dataset[target]
y = y.map({"Iris-setosa":0 , "Iris-versicolor" : 1 , "Iris-virginica": 2})

In [27]:
scalar = StandardScaler()
x_train , x_test , y_train , y_test = train_test_split(
    x, y, test_size=0.25, random_state=42
)
x_train += np.random.normal(0, 0.1, x_train.shape)

x_train = scalar.fit_transform(x_train)
x_test  = scalar.transform(x_test)

X_train_tensor = torch.tensor(x_train , dtype=torch.float32)
Y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)

X_test_tensor = torch.tensor(x_test , dtype=torch.float32)
Y_test_tensor = torch.tensor(y_test.values , dtype=torch.long)

In [30]:
model = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 24),
    nn.ReLU(),

    nn.Linear(24, 12),
    nn.ReLU(),

    nn.Linear(12, 6),
    nn.ReLU(),

    nn.Linear(6, 3)   # output = 3 classes
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(400):
    y_pred = model(X_train_tensor)
    
    loss = criterion(y_pred , Y_train_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")
        
        
with torch.no_grad():
    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs, 1)

    correct = (predicted == Y_test_tensor).sum().item()
    acc = correct / len(Y_test_tensor)

print("Test Accuracy:", acc)
with torch.no_grad():
    train_out = model(X_train_tensor)
    _, train_pred = torch.max(train_out, 1)
    train_acc = (train_pred == Y_train_tensor).sum().item() / len(Y_train_tensor)

print("Train Accuracy:", train_acc)

Epoch 0, Loss: 1.131103277206421
Epoch 50, Loss: 1.0080015659332275
Epoch 100, Loss: 0.7751470804214478
Epoch 150, Loss: 0.5830296277999878
Epoch 200, Loss: 0.43813052773475647
Epoch 250, Loss: 0.35382652282714844
Epoch 300, Loss: 0.30849456787109375
Epoch 350, Loss: 0.27912089228630066
Test Accuracy: 1.0
Train Accuracy: 1.0


In [ ]:
joblib.dump(model , "iris_nn.pkl")